In [0]:
# Gold Layer - Star Schema building with incremental load
# Reads from Silver Delta table and builds dimension and fact tables using Delta MERGE

In [0]:
# Configuration

storage_account_name = dbutils.secrets.get(scope="etl_sales_scope", key="STORAGE_ACCOUNT_NAME")
storage_account_key = dbutils.secrets.get(scope="etl_sales_scope", key="STORAGE_ACCOUNT_KEY")
container_name = dbutils.secrets.get(scope="etl_sales_scope", key="AZURE_CONTAINER_NAME")

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

base_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net"
silver_path = f"{base_path}/silver/sales"
gold_path = f"{base_path}/gold"

print("Configuration loaded successfully")

In [0]:
# Read Silver layer

df_silver = spark.read.format("delta").load(silver_path)
print(f"Silver table loaded: {df_silver.count()} rows")


In [0]:
# Build dimension tables

from pyspark.sql.functions import monotonically_increasing_id, current_timestamp

# dim_product
def build_dim_product(df):
    return df.select('product_category') \
             .distinct() \
             .withColumn('product_id', monotonically_increasing_id() + 1) \
             .withColumn('created_at', current_timestamp()) \
             .withColumn('updated_at', current_timestamp())

# dim_region
def build_dim_region(df):
    return df.select('customer_region') \
             .distinct() \
             .withColumn('region_id', monotonically_increasing_id() + 1) \
             .withColumn('created_at', current_timestamp()) \
             .withColumn('updated_at', current_timestamp())

# dim_payment
def build_dim_payment(df):
    return df.select('payment_method') \
             .distinct() \
             .withColumn('payment_method_id', monotonically_increasing_id() + 1) \
             .withColumn('created_at', current_timestamp()) \
             .withColumn('updated_at', current_timestamp())

# dim_date
def build_dim_date(df):
    return df.select(
        'order_date', 'day_num', 'month_num',
        'quarter_num', 'year_num', 'day_of_week'
    ).distinct() \
     .withColumn('date_id', monotonically_increasing_id() + 1)

dim_product = build_dim_product(df_silver)
dim_region = build_dim_region(df_silver)
dim_payment = build_dim_payment(df_silver)
dim_date = build_dim_date(df_silver)

print("Dimension tables built successfully")